# 01 · EDA dirigido al modelado — Recepción de juegos en Steam
**Proyecto ML (Project Break II).** Objetivo: predecir si un juego tendrá **recepción positiva** del público a partir de sus características.

- **Target:** `recepcion_positiva` (1 si el % de reseñas positivas ≥ 70%, entre juegos con reseñas suficientes).
- **Tipo:** clasificación binaria. **Métrica:** F1 / ROC-AUC (clases desbalanceadas).
- **Dataset:** Steam Games Dataset (FronkonGames), 122.611 juegos × 39 columnas.

> Nota: este EDA está *dirigido al modelado*: cada gráfica responde a una pregunta útil para construir el modelo, no es exploración porque sí.

## 1. Carga
Descarga `games.csv` y ponlo en `src/data/`. Cargamos solo las columnas útiles para no arrastrar los campos de texto pesados (descripciones, URLs, capturas…).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

usecols = ['AppID','Name','Release date','Estimated owners','Peak CCU','Required age','Price',
           'DiscountDLC count','Supported languages','Windows','Mac','Linux','Metacritic score',
           'User score','Positive','Negative','Achievements','Recommendations',
           'Average playtime forever','Developers','Publishers','Categories','Genres','Tags']

df = pd.read_csv('../data/games.csv', encoding='utf-8', usecols=usecols)
df.shape

## 2. Primer vistazo
Forma, tipos, nulos y primeras filas: entender qué tenemos antes de definir nada.

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df.head()

## 3. Definición de la variable objetivo
No hay una columna "buena acogida" ya hecha: la construimos a partir de `Positive` y `Negative`.

- `total_reviews = Positive + Negative`.
- Nos quedamos solo con juegos con **suficientes reseñas** (≥ 50) para que la etiqueta sea fiable (un juego con 2 reseñas no dice nada).
- `pct_pos = Positive / total_reviews`.
- `recepcion_positiva = 1` si `pct_pos ≥ 0.70`, si no `0`. (El umbral 0.70 es ajustable; Steam usa ~70% para "Mostly Positive").

In [ ]:
df['total_reviews'] = df['Positive'] + df['Negative']
reviewed = df[df['total_reviews'] >= 50].copy()
reviewed['pct_pos'] = reviewed['Positive'] / reviewed['total_reviews']
reviewed['recepcion_positiva'] = (reviewed['pct_pos'] >= 0.70).astype(int)

print('Juegos totales:', len(df))
print('Juegos con >=50 reseñas (los que usaremos):', len(reviewed))
print('Balance del target:')
print(reviewed['recepcion_positiva'].value_counts(normalize=True).round(3))

### 3.1 Análisis del target
El % de reseñas positivas y el balance de clases. **Hallazgo:** ~75% de los juegos con reseñas son "positivos" → **clases desbalanceadas (75/25)**. Por eso NO usaremos accuracy como métrica principal, sino **F1 / ROC-AUC**, y valoraremos técnicas de balanceo (class_weight, etc.).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].hist(reviewed['pct_pos'], bins=40, color='#4c72b0', edgecolor='white')
ax[0].axvline(0.70, color='red', ls='--', label='umbral 70%')
ax[0].set_title('Distribución del % de reseñas positivas'); ax[0].set_xlabel('% positivas'); ax[0].legend()

reviewed['recepcion_positiva'].value_counts().sort_index().plot(
    kind='bar', ax=ax[1], color=['#c44e52','#55a868'])
ax[1].set_title('Balance de clases del target'); ax[1].set_xticklabels(['No (0)','Sí (1)'], rotation=0)
plt.tight_layout(); plt.show()

## 4. Primeras relaciones feature → target (a explorar)
Ideas de qué mirar en las siguientes celdas (cada una es una pregunta para el modelo):
- **Precio vs recepción:** ¿los juegos más caros gustan más? (`Price` por clase).
- **Modelo de negocio:** F2P (Price == 0) vs de pago.
- **Plataformas:** ¿multiplataforma (Windows/Mac/Linux) se asocia a mejor recepción?
- **Género principal** (primer valor de `Genres`) y **etiquetas** (`Tags`).
- **Metacritic / achievements / DLC** como señales de calidad.

Ejemplo para empezar (precio por clase):

In [ ]:
import warnings; warnings.filterwarnings('ignore')
reviewed['es_f2p'] = (reviewed['Price'] == 0).astype(int)
print(reviewed.groupby('recepcion_positiva')['Price'].median())
print(reviewed.groupby('recepcion_positiva')['es_f2p'].mean())

## 5. Próximos pasos
1. Completar el EDA de features (precio, género, plataformas, idiomas, Metacritic…).
2. Preprocesado: limpiar `Estimated owners` (rango de texto), parsear `Supported languages`, extraer `genero_principal` de `Genres`, separar la columna mal formada `DiscountDLC count`, tratar nulos.
3. Split train/test **antes** del preprocesado pesado (evitar data leakage).
4. Modelado: baseline → comparativa → optimización → evaluación.

> Decisiones de esta fase: target = recepción positiva (≥70%), solo juegos con ≥50 reseñas, métrica F1/ROC-AUC por el desbalanceo.